# 读取 train / test

In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import pandas as pd
import numpy as np
# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [26]:
# 读数据
import pandas as pd

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(train.shape)
print(test.shape)

train.head()

(668665, 15)
(286571, 14)


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [27]:
# 查看所有列名
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

# 自动找出 target
target_col = list(set(train.columns) - set(test.columns))
print("\nTarget column:", target_col)

Train columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV']

Test columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Target column: ['Will_Buy_EV']


In [28]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  str    
 12  Subsidy_Available            668665 non-null  str    
 13  Range_Anxi

In [29]:
target = target_col[0]

print(train[target].value_counts())
print(train[target].value_counts(normalize=True))

Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64
Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


---
# 第一个 baseline 模型

In [30]:
categorical_cols = train.select_dtypes(include='object').columns.tolist()

# 去掉 target
categorical_cols.remove('Will_Buy_EV')

for col in categorical_cols:
    print(f"\n{col}:")
    print(train[col].value_counts())


Gender:
Gender
Male      367954
Female    295427
Other       5284
Name: count, dtype: int64

City_Type:
City_Type
Urban       289305
Suburban    255377
Rural       123983
Name: count, dtype: int64

Current_Car_Type:
Current_Car_Type
Sedan        303459
SUV          246545
Hatchback     79438
Truck         39223
Name: count, dtype: int64

Home_Charging_Possible:
Home_Charging_Possible
Yes    462677
No     205988
Name: count, dtype: int64

Subsidy_Available:
Subsidy_Available
Yes    419909
No     248756
Name: count, dtype: int64

Range_Anxiety_Level:
Range_Anxiety_Level
Low       603972
Medium     62499
High        2194
Name: count, dtype: int64


/var/folders/sn/52nckwwx6qnfvj3j53ch2lj00000gn/T/ipykernel_85287/4225261881.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train.select_dtypes(include='object').columns.tolist()


In [31]:
test.isnull().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
dtype: int64

### 2.准备 X 和 y

In [32]:
target = 'Will_Buy_EV'

# 目标变量：Yes -> 1, No -> 0
y = train[target].map({
    'Yes': 1,
    'No': 0
})

# 第一版先去掉 id 和 target
X = train.drop(columns=['id', target])

print(X.shape)
print(y.shape)

X.head()

(668665, 13)
(668665,)


,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low


### 3.划分训练集和验证集

// 把训练数据分成：
80% → training
20% → validation

把单次 80/20 划分换成 5-Fold CV

### 4.找 categorical columns

In [33]:
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(categorical_features)

['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


/var/folders/sn/52nckwwx6qnfvj3j53ch2lj00000gn/T/ipykernel_85287/1371258272.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include='object').columns.tolist()


### 5.训练第一个 CatBoost

In [34]:
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# test features
X_test = test.drop(columns=['id'])

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

fold_scores = []

feature_importance = np.zeros(X.shape[1])

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f"\n========== Fold {fold + 1} ==========")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=7,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42 + fold,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=150
    )

    feature_importance += model.feature_importances_ / 5

    valid_pred = model.predict_proba(X_valid)[:, 1]

    oof_pred[valid_idx] = valid_pred

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(fold_auc)

    print("Fold AUC:", fold_auc)

    test_pred += model.predict_proba(X_test)[:, 1] / 5

print("\nFold scores:", fold_scores)
print("Mean Fold AUC:", np.mean(fold_scores))
print("OOF AUC:", roc_auc_score(y, oof_pred))


========== Fold 1 ==========
0:	test: 0.9311980	best: 0.9311980 (0)	total: 185ms	remaining: 6m 8s
200:	test: 0.9386884	best: 0.9386884 (200)	total: 20.5s	remaining: 3m 3s
400:	test: 0.9392814	best: 0.9392814 (400)	total: 41.8s	remaining: 2m 46s
600:	test: 0.9397808	best: 0.9397808 (600)	total: 1m 3s	remaining: 2m 26s
800:	test: 0.9400559	best: 0.9400559 (800)	total: 1m 26s	remaining: 2m 10s
1000:	test: 0.9402578	best: 0.9402578 (1000)	total: 1m 49s	remaining: 1m 49s
1200:	test: 0.9403592	best: 0.9403592 (1200)	total: 2m 14s	remaining: 1m 29s
1400:	test: 0.9404296	best: 0.9404296 (1400)	total: 2m 38s	remaining: 1m 7s
1600:	test: 0.9404976	best: 0.9404982 (1596)	total: 3m 2s	remaining: 45.6s
1800:	test: 0.9405102	best: 0.9405127 (1796)	total: 3m 26s	remaining: 22.8s
1999:	test: 0.9405206	best: 0.9405251 (1988)	total: 3m 51s	remaining: 0us

bestTest = 0.9405250587
bestIteration = 1988

Shrink model to first 1989 iterations.
Fold AUC: 0.9405250587111721

========== Fold 2 ==========
0:	te

In [35]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
})

importance = importance.sort_values(
    'Importance',
    ascending=False
)

importance

,Feature,Importance
11,Subsidy_Available,53.070957
6,Environmental_Concern_Level,19.670366
1,Annual_Income_USD,10.095504
12,Range_Anxiety_Level,3.818784
0,Age,3.333003
2,Daily_Commute_km,3.312854
4,Charging_Stations_Near_Home,1.803916
5,Charging_Stations_Near_Work,1.741677
10,Home_Charging_Possible,0.942030
9,Current_Car_Type,0.785810


In [36]:
X_test = test.drop(columns=['id'])

print(test_pred[:10])
print("Number of predictions:", len(test_pred))

[9.62409744e-03 2.07689752e-02 4.33602923e-03 2.51813713e-03
 2.36625238e-02 4.14785105e-03 3.66084919e-01 5.21022534e-05
 6.56013383e-01 6.37241981e-03]
Number of predictions: 286571


# 生成 submission

In [37]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# 确保全部折训练完成，避免误用中断后的预测
assert len(fold_scores) == skf.get_n_splits()
assert len(test_pred) == len(test)
assert np.isfinite(test_pred).all()
assert ((test_pred >= 0) & (test_pred <= 1)).all()

submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "Will_Buy_EV": test_pred,
})
submission.to_csv(output_dir / "submission.csv", index=False)